In [2]:
import os, json
import pandas as pd
import openai
from dotenv import load_dotenv
from pathlib import Path
from tqdm import tqdm
import time

In [3]:
# Load API
load_dotenv()
openai.api_key = os.getenv("OPENAI_API_KEY")
MODEL_NAME     = "gpt-4o-mini-2024-07-18" 
TEMPERATURE    = 1.0
print("API Key loaded:", openai.api_key is not None)

# Load files
src_file  = "../data/disaster_description.csv"
out_file  = "extracted_twostage.csv"
df = pd.read_csv(src_file)
# df = df.head(200)  # For test

API Key loaded: True


# Materials

In [5]:
DISASTER_CLASSIFICATION = json.load(open("../data/classification_document.json"))

In [6]:
MAGNITUDE_INFORMATION = json.load(open("../data/magnitude_document.json"))

In [7]:
FIELD_SCHEMA = json.load(open("../data/schema_document.json"))

In [8]:
CATEGORICAL_VALUES = {
    "Disaster Classification": DISASTER_CLASSIFICATION,
    "Magnitude": MAGNITUDE_INFORMATION
}

# Prompt

In [10]:
OUTPUT_FIELDS = {
  "Event": "fields name",
  "Geographical Information": "fields name",
  "Effect": "fields name"
}

In [11]:
OUTPUT_VALUES = {
  "Event": {"fields": "values"},
  "Geographical Information": {"fields": "values"},
  "Effect": {"fields": "values"}
}

In [12]:
fields_extraction_prompt = f"""
        Please refer FIELD_SCHEMA first, then list all fields that are marked as 'Required' or are mentioned in DISASTER_DESCRIPTION. 
        Output must contain field names only.
        Finally return a single JSON object that conforms precisely to OUTPUT_FIELDS.
        
        FIELD_SCHEMA:
        {json.dumps(FIELD_SCHEMA, ensure_ascii=False)}

        OUTPUT_FIELDS:
        {json.dumps(OUTPUT_FIELDS, ensure_ascii=False)}

    """

In [13]:
values_extraction_prompt = f"""
        Please extract all information regarding only the fields in EXTRACTED_FIELDS 
        from text DISASTER_DESCRIPTION (refering the information in FIELD_SCHEMA and using only 
        the categorical options in CATEGORICAL_VALUES). 
        Finally return a single JSON object that conforms precisely to OUTPUT_VALUES.
        
        FIELD_SCHEMA:
        {json.dumps(FIELD_SCHEMA, ensure_ascii=False)}

        CATEGORICAL_VALUES:
        {json.dumps(CATEGORICAL_VALUES, ensure_ascii=False)}

        OUTPUT_VALUES:
        {json.dumps(OUTPUT_VALUES, ensure_ascii=False)}
    """


# Implement

In [15]:
def call_llm(prompt_text: str, description_text: str, extracted_fields_text = None) -> dict:
    
    if extracted_fields_text is None:
        user_content = description_text or ""
    else:
        user_content = json.dumps({
            "DISASTER_DESCRIPTION": description_text,
            "EXTRACTED_FIELDS": extracted_fields_text
        }, ensure_ascii=False)
        
    try:
        resp = openai.chat.completions.create(
            model           = MODEL_NAME,
            messages        = [
                {"role": "system", "content": prompt_text},
                {"role": "user", "content": user_content}
            ],
            temperature     = TEMPERATURE,
            response_format = {"type": "json_object"}
        )
        return json.loads(resp.choices[0].message.content)
    except Exception as e:
        print("LLM Error:", e)
        return None

In [16]:
fields_results = []
values_results = []
for desc in tqdm(df["description"], desc="Two-stage extracting"):
    # Stage1: fields
    p1_resp       = call_llm(fields_extraction_prompt, desc)

    fields_results.append(json.dumps(p1_resp, ensure_ascii=False)) 
    
    # Stage2: values
    p2_resp       = call_llm(values_extraction_prompt, desc, p1_resp)
    values_results.append(p2_resp)
    
    time.sleep(0.5)

Two-stage extracting:  21%|███▎            | 327/1588 [19:04<1:14:11,  3.53s/it]

LLM Error: Out of range float values are not JSON compliant
LLM Error: Out of range float values are not JSON compliant


Two-stage extracting:  23%|███▌            | 359/1588 [21:00<1:18:56,  3.85s/it]

LLM Error: Out of range float values are not JSON compliant
LLM Error: Out of range float values are not JSON compliant


Two-stage extracting: 100%|███████████████| 1588/1588 [1:30:02<00:00,  3.40s/it]


In [17]:
df["two_stage_fields"] = fields_results
df["two_stage_values"] = values_results

df.to_csv(out_file, index=False)
print(f"Saved to {out_file}")

Saved to extracted_twostage.csv
